In [1]:
import re

import pandas as pd
import spacy

nlp = spacy.load("en_core_web_trf")

In [2]:
df = pd.read_csv(
    "./../temp_error_analyis/label_temp_answers_head_dataset/data/notebook_output/03_label_answers_with_year_timeunit_head_data.csv"
).query("answer_timeunit == 'year'")
df.head()

,question,answer,table_id,answer_type,is_temporal,answer_old,answer_timeunit
0,How many years did Art Carney as actor since 1...,54 Years,2,TEMPORAL,Is temporal,54 Years,year
1,How many total years was Art Carney married to...,28 years,2,TEMPORAL,Is temporal,28 years,year
2,How many years before he died was Art Carney m...,23,2,COUNT,Is temporal,23,year
3,How old was Art Carney when he first got divor...,47,2,AGE,Is temporal,47,year
4,How many years ago did Art Carney was died?,19 Years ago,2,TEMPORAL,Is temporal,19 Years ago,year


In [3]:
flant5 = pd.read_csv(
    "./../../models/predictions/flant5/flant5_xxl/fewshot_with_reasoning/indomain_eval_flant5_xxl_few_shot_reasoning.csv"
).drop(columns="Unnamed: 0")
flant5.shape

(1901, 4)

In [4]:
df.merge(flant5, on="question", how="inner").shape

(859, 10)

In [5]:
word_to_digit = {
    "zero": "0",
    "one": "1",
    "two": "2",
    "three": "3",
    "four": "4",
    "five": "5",
    "six": "6",
    "seven": "7",
    "eight": "8",
    "nine": "9",
    "ten": "10",
}

pipe_between_words = "|".join(word_to_digit.keys())
pattern = re.compile(r"\b(" + pipe_between_words + r")\b", re.IGNORECASE)

# Ignore case and lower because some digits are writing with capital letters
def replace_num_word(match):
    word = match.group(1)  # The matched word (e.g. "one")
    return word_to_digit[word.lower()]  # The corresponding digit (e.g. "1")

In [6]:
temp_err_df = (
    df.merge(
        flant5,
        left_on=[
            "question",
            "answer_old",
        ],  # answer_old is unmodified, i.e., contains digits still as words ("three" instead of "3")
        right_on=["question", "actual_answer"],
        how="inner",
    )
    .drop(columns=["table", "answer_timeunit", "is_temporal", "answer_old", "actual_answer", "answer_type", "table_id"])
    .assign(
        predicted_answer_old=lambda x: x["predicted_answer"],
        predicted_answer=lambda x: x["predicted_answer_old"].apply(
            lambda y: pattern.sub(replace_num_word, y)
        ),
        answer_digits=lambda x: x["answer"].str.findall("\d+"),
        predicted_answer_digits=lambda x: x["predicted_answer"].str.findall("\d+"),
    )
)
temp_err_df.head()

,question,answer,predicted_answer,predicted_answer_old,answer_digits,predicted_answer_digits
0,How many years did Art Carney as actor since 1...,54 Years,Art Carney was an actor from 1939 to 1993. The...,Art Carney was an actor from 1939 to 1993. The...,[54],"[1939, 1993, 54]"
1,How many total years was Art Carney married to...,28 years,Jean Myers and Art Carney were married for a t...,Jean Myers and Art Carney were married for a t...,[28],"[40, 65]"
2,How many years before he died was Art Carney m...,23,The last time Art Carney was married was in 19...,The last time Art Carney was married was in 19...,[23],"[1977, 2003]"
3,How old was Art Carney when he first got divor...,47,The first marriage of Art Carney was to Jean M...,The first marriage of Art Carney was to Jean M...,[47],[1940]
4,How many years ago did Art Carney was died?,19 Years ago,"Art Carney died on November 9, 2003, at the ag...","Art Carney died on November 9, 2003, at the ag...",[19],"[9, 2003, 85, 15]"


In [7]:
temp_err_single_digit = temp_err_df.query("answer_digits.str.len()==1").assign(
    expected_minus_pred=lambda x: x["answer_digits"].apply(set)
    - x["predicted_answer_digits"].apply(set),
    has_overlap=lambda x: x["expected_minus_pred"].apply(len) == 0,
)
temp_err_single_digit.shape

(855, 8)

In [8]:
temp_err_single_digit.query("answer_digits.str.len()==1 and answer_digits==predicted_answer_digits").shape

(53, 8)

In [9]:
temp_err_single_digit.query("has_overlap == True").shape

(184, 8)

In [10]:
temp_err_single_digit.query("has_overlap != True and answer_digits!=predicted_answer_digits").shape

(671, 8)

In [11]:
temp_err_single_digit.query("has_overlap != True and answer_digits!=predicted_answer_digits").loc[
    :, ["question", "predicted_answer", "answer_digits", "predicted_answer_digits", "has_overlap"]
]

,question,predicted_answer,answer_digits,predicted_answer_digits,has_overlap
1,How many total years was Art Carney married to...,Jean Myers and Art Carney were married for a t...,[28],"[40, 65]",False
2,How many years before he died was Art Carney m...,The last time Art Carney was married was in 19...,[23],"[1977, 2003]",False
3,How old was Art Carney when he first got divor...,The first marriage of Art Carney was to Jean M...,[47],[1940],False
4,How many years ago did Art Carney was died?,"Art Carney died on November 9, 2003, at the ag...",[19],"[9, 2003, 85, 15]",False
5,For how many years had Benedict Cumberbatch be...,Benedict Cumberbatch has been acting for 15 ye...,[17],"[15, 2015]",False
...,...,...,...,...,...
851,How many years difference was there between th...,reasoning. The answer is : First T20I v | West...,[40],[20],False
852,How long did Sri Lanka take from being an ICC ...,1965 | ) Best result Quarter-finalist (2019-2021),[16],"[1965, 2019, 2021]",False
853,How long did it take for Sri Lanka to become a...,Sri Lanka was an Associate Member of the ICC f...,[16],"[1965, 1981, 6]",False
855,How long after Sri Lanka's first ODI and first...,1975 | ) T20Is First T20I v | West Indies at,[31],"[1975, 20, 20]",False


In [12]:
# Reference for definitions: https://www.newscatcherapi.com/blog/named-entity-recognition-with-spacy
ENTS_OF_INTEREST = [
    "CARDINAL",
    "TIME",
    "DATE"
]

In [13]:
temp_err_single_digit = temp_err_single_digit.assign(
    predicted_answer_nlp=lambda x: x["predicted_answer"].apply(nlp),
    answer_nlp=lambda x: x["answer"].apply(nlp),
    predicted_answers_ents=lambda x: x["predicted_answer_nlp"].apply(
        lambda doc: [ent for ent in doc.ents if ent.label_ in ENTS_OF_INTEREST]
    ),
    predicted_answers_ent_labels=lambda x: x["predicted_answers_ents"].apply(
        lambda ents: [ent.label_ for ent in ents]
    ),
    answers_ents=lambda x: x["answer_nlp"].apply(lambda doc: [ent for ent in doc.ents]),
    answers_ent_labels=lambda x: x["answers_ents"].apply(lambda ents: [ent.label_ for ent in ents]),
)

In [51]:
temp_err_single_digit_small = (
    temp_err_single_digit.drop(
        columns=[
            # "question",
            "predicted_answer_old",
            "predicted_answer_nlp",
            "answer_nlp",
            "expected_minus_pred",
        ]
    )
    .query("has_overlap == False")
    .drop(columns="has_overlap")
)
temp_err_single_digit_small.loc[2].predicted_answer

'The last time Art Carney was married was in 1977. He died in 2003. So,'

## Pos
- Q: 'How many years did Art Carney as actor since 1939?'
- P: 'Art Carney was an actor from 1939 to 1993. The answer is 54.'
- E: '54 Years'

## Neg but right temporal unit
- Q: 'How long did it take for Sri Lanka to become a full member after attaining its Associate Member ICC status?'
- P: 'Sri Lanka was an Associate Member of the ICC from 1965 to 1981. The answer is 6.'
- E: '16 years'

## Entirely false
- Q: 'How many years before he died was Art Carney married for the last time?'
- P: 'The last time Art Carney was married was in 1977. He died in 2003. So,'
- E: '23'

In [24]:
temp_err_single_digit_small.assign(
    ent_overlap=lambda x: x["predicted_answers_ent_labels"].apply(set)
    & x["answers_ent_labels"].apply(set)
).query("ent_overlap == False").shape

(128, 9)

In [57]:
def flatten(xss):
    return [x for xs in xss for x in xs]

In [61]:
temp_err_single_digit.loc[:, ["question", "answer", "predicted_answer"]].to_csv("flant5_eval.csv", index=False)

In [ ]:
# Examples for testing on cluster
temp_err_single_digit.loc[[3,4, 13]]

array(['Dwayne Johnson retired in 2019 after a career that lasted 23 years. The'],
      dtype=object)

In [79]:
temp_err_single_digit.query("has_overlap==False").loc[4].predicted_answer

'Art Carney died on November 9, 2003, at the age of 85. The answer is 15'